# P1: Verifier-Aware Learned Controller — Colab GPU Pipeline

**Runtime:** `Runtime → Change runtime type → T4 GPU` (or A100)

Runs Phase 1 (baselines) → Phase 2 (features) → Phase 3 (train + eval) on GPU.

Upload this `p01-verifier-aware-controller` folder to Colab, or mount Drive (see `colab/README.md`).

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. Get project code

Pick **one** method below (uncomment the block you need).

In [ ]:
# --- Method A: Already uploaded / in Drive (edit path) ---
import os
PROJECT = '/content/p01-verifier-aware-controller'  # change if using Drive

# --- Method B: Mount Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT = '/content/drive/MyDrive/Research/p01-verifier-aware-controller'

# --- Method C: Upload zip ---
# from google.colab import files
# uploaded = files.upload()
# !unzip -q -o p01-verifier-aware-controller.zip -d /content/
# PROJECT = '/content/p01-verifier-aware-controller'

assert os.path.isdir(PROJECT), f'Project not found: {PROJECT}'
%cd {PROJECT}
!ls -la scripts/ configs/

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q huggingface_hub accelerate

## 3. Hugging Face login (required for real Qwen runs)

Skip for `--smoke-test` only.

In [ ]:
from huggingface_hub import login
login()  # paste HF token when prompted

## 4. Configuration

| Variable | Smoke | Full Colab run |
|---|---|---|
| `SMOKE` | `True` | `False` |
| `MAX_EXAMPLES` | ignored | `200` (increase on A100) |
| `LABEL_N` | `4` | `8` |

In [ ]:
SMOKE = False          # True = fast mock, no model download
MAX_EXAMPLES = 200     # per benchmark cap for Colab
LABEL_N = 8            # samples per query for Phase 2 labels
SAVE_TO_DRIVE = False  # set True + mount Drive to persist results
DRIVE_OUT = '/content/drive/MyDrive/Research/p01_colab_results'

## 5. Run full pipeline (Phases 1 → 2 → 3)

In [ ]:
import subprocess, sys

cmd = [sys.executable, 'scripts/colab_run_all.py', '--config', 'configs/colab.yaml']
if SMOKE:
    cmd.append('--smoke-test')
else:
    cmd += ['--max-examples', str(MAX_EXAMPLES), '--label-n', str(LABEL_N)]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## 6. View results

In [ ]:
import json
from IPython.display import Image, display

base = 'results/colab_run' if not SMOKE else 'results/phase3_smoke'

for path in [
    f'{base}/phase1/phase1_summary.json',
    f'{base}/phase3/phase3_eval_summary.json',
    f'{base}/phase3/phase3_train_summary.json',
]:
    try:
        with open(path) as f:
            print('\n===', path, '===')
            print(json.dumps(json.load(f), indent=2)[:4000])
    except FileNotFoundError:
        print('Missing:', path)

for img in [f'{base}/phase1/phase1_pareto.png', f'{base}/phase3/phase3_comparison.png']:
    try:
        display(Image(filename=img))
    except Exception:
        pass

## 7. (Optional) Save to Google Drive

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil, datetime
    stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
    dest = f'{DRIVE_OUT}_{stamp}'
    shutil.copytree('results/colab_run', dest, dirs_exist_ok=True)
    print('Saved to', dest)

## Individual phases (manual)

```bash
# Phase 1 baselines only
!python scripts/run_phase1_baselines.py --config configs/colab.yaml --max-examples 200

# Phase 2 extract + train features
!python scripts/run_phase2.py --mode both --config configs/colab.yaml --max-examples 200 --force

# Phase 3 ablations + eval
!python scripts/run_phase3.py --mode both --config configs/phase3.yaml --max-examples 200
```